# Day 1 — What is an Agent? Build ReAct from Scratch

---

Today, in about 75 minutes, you'll answer three questions:

1. What actually makes an "AI agent" different from a plain LLM call?
2. What is the **ReAct** loop — the pattern behind nearly every agent?
3. Can I write one from scratch in ~50 lines of Python? (Yes.)

**No framework today.** We deliberately skip LangGraph / CrewAI so you see the mechanics. Framework starts Day 3.


## 1. Agent = LLM in a loop with tools

A regular LLM call:

```
   question  ->  [LLM]  ->  answer     (done)
```

An agent:

```
   question  ->  [LLM]  ->  "I need to search the web for X"
                                │
                                ▼
                        [call web_search tool]
                                │
                                ▼
                         result: "..."
                                │
                                ▼
                             [LLM]  ->  "Now I need to fetch this URL"
                                │
                                ▼
                        [call fetch tool]
                                │
                                ▼
                             [LLM]  ->  answer  (done)
```

The LLM keeps **thinking, acting, observing** until it decides it has the answer. That's the loop.

Three ingredients:
- **Perception** — the LLM reads the current state (question + past steps).
- **Planning + Action** — it decides "call tool X with args Y" or "I'm done."
- **Observation** — the tool result feeds back into the next thought.


## 2. The ReAct pattern (Reason + Act)

**ReAct** is the simplest agent pattern that still works well. Prompt the LLM to output structured "Thought / Action / Observation" turns:

```
Thought: I need to know the population of Tokyo.
Action: web_search[population of Tokyo]
Observation: Tokyo has ~13.9 million people.
Thought: Now I can answer.
Action: finish[Tokyo has about 13.9 million people.]
```

You parse the `Action:` line, run the tool, paste the result as `Observation:`, and send the growing transcript back to the LLM. Repeat until you see `finish[...]`.

That's the *entire* algorithm. Everything sophisticated in modern agent frameworks is polish on top of this loop.


## 3. Setup


In [ ]:
!pip install together python-dotenv --quiet

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
assert os.getenv("TOGETHER_API_KEY"), "Set TOGETHER_API_KEY in .env"

from together import Together
llm = Together()


## 4. Two tools — calculator and lookup

We'll give our agent two toy tools so it has real reasons to call them.


In [ ]:
# A safe calculator (no eval on arbitrary strings — restricted grammar)
import ast, operator

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub,
        ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.USub: operator.neg}

def _eval(node):
    if isinstance(node, ast.Num): return node.n
    if isinstance(node, ast.BinOp): return _OPS[type(node.op)](_eval(node.left), _eval(node.right))
    if isinstance(node, ast.UnaryOp): return _OPS[type(node.op)](_eval(node.operand))
    raise ValueError("unsupported expression")

def calc(expr: str) -> str:
    try:
        return str(_eval(ast.parse(expr, mode="eval").body))
    except Exception as e:
        return f"error: {e}"


# A tiny "database" lookup
FACTS = {
    "acmecloud pro price": "$29 per month",
    "acmecloud free tier": "10 GB storage, 100 API calls/day",
    "acmecloud founders":  "Priya Rao and Marcus Chen, 2019",
}
def lookup(key: str) -> str:
    return FACTS.get(key.lower().strip(), "not found")

print(calc("(29 * 12) + 100"))
print(lookup("AcmeCloud Pro price"))


## 5. The ReAct system prompt

We tell the LLM the exact output format we expect.


In [ ]:
SYSTEM = """You are a ReAct agent. On every turn respond with EXACTLY one of:

Thought: <your reasoning>
Action: calc[<math expression>]
Action: lookup[<key to look up>]
Action: finish[<final answer to the user>]

Rules:
- Emit ONE action per turn. Do not chain.
- After you see Observation: text on the next turn, decide what to do next.
- When you have the final answer, use finish[...] and stop.
"""


## 6. The loop — the whole agent, ~30 lines


In [ ]:
import re

ACTION_RE = re.compile(r"Action:\s*(calc|lookup|finish)\[(.*?)\]", re.DOTALL)


def run_tool(name: str, arg: str) -> str:
    if name == "calc":   return calc(arg)
    if name == "lookup": return lookup(arg)
    return f"unknown tool: {name}"


def agent(question: str, max_steps: int = 6) -> str:
    transcript = [
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": f"Question: {question}"},
    ]
    for step in range(max_steps):
        resp = llm.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=transcript,
            temperature=0.0,
            stop=["Observation:"],
        )
        turn = resp.choices[0].message.content.strip()
        print(f"\n--- step {step+1} ---\n{turn}")

        m = ACTION_RE.search(turn)
        if not m:
            return "(agent stopped — could not parse an Action)"

        name, arg = m.group(1), m.group(2).strip()
        if name == "finish":
            return arg

        obs = run_tool(name, arg)
        print(f"Observation: {obs}")

        transcript.append({"role": "assistant", "content": turn})
        transcript.append({"role": "user",      "content": f"Observation: {obs}"})

    return "(agent stopped — max steps reached)"


answer = agent("If I pay for AcmeCloud Pro for 2 years, how much is that in total?")
print("\n=== FINAL ===")
print(answer)


**Trace what happened:**

1. LLM thought "I need the Pro price" → called `lookup`.
2. Got `"$29 per month"`, thought "24 months × 29" → called `calc`.
3. Got `696` → finished.

The magic isn't in any single call. It's in the **loop + tool observations feeding back into context.**


## 7. Why `stop=["Observation:"]` matters

We told the LLM the format includes `Observation: <text>` on the next turn. Without a stop sequence, the model would happily *hallucinate its own observations* — inventing tool results.

**Setting `stop=["Observation:"]` forces the model to end its turn before the next observation.** We control what actually goes into that observation slot. This is a critical pattern for every from-scratch agent.


## 8. Common agent patterns (know the names)

You'll hear these in job interviews. All are variations of the ReAct loop:

| Pattern | Idea | When to use |
|---|---|---|
| **ReAct** | Thought → Action → Observation, repeat | Default. What we just built. |
| **Plan-and-Execute** | Draft the full plan first, then execute steps | Long, multi-stage tasks (research, coding) |
| **Reflexion** | After failing, agent writes a "lesson learned" for its next try | Tasks with clear success/failure signal |
| **Chain-of-Thought (CoT)** | Just "think step-by-step" — no tools | Not really an agent; pure LLM reasoning |

Know the names, use ReAct 95% of the time. Frameworks like LangGraph let you switch patterns as your task grows.


## Recap

- An **agent** = LLM in a loop with **tools** it decides to call.
- **ReAct** = the simplest working pattern: Thought → Action → Observation.
- You can build one in ~30 lines. **No framework needed** to understand it.
- **`stop` sequences** keep the LLM from hallucinating tool results.
- **max_steps** is your safety net — always cap the loop.
- **Next class:** better tools — real web search, HTTP fetch, and OpenAI's structured function-calling API.
